# Notebook 01 — Normal facial skin: SDXL + optional texture LoRA + custom LoRA + Compel

**Our acne-style research workflow, adapted to the normal-skin class.** The same four comparison modes are supported:

- `base`: SDXL Base 1.0 alone
- `texture`: SDXL + `profpeng/skinenhancer` (optional, not medically validated and acne-oriented)
- `custom`: SDXL + your trained normal-face LoRA from Notebook 02
- `combined`: SDXL + both LoRAs, as an **experimental** comparison

For each selected mode and each editable morphology/photo prompt, generate **five images** with seeds matched across modes. Use Compel 2.3.1 with SDXL's dual text encoders, **no long-prompt truncation**, actual dual-tokenizer validation (max 150 content tokens each), equal-length positive/negative embeddings, and pooled embeddings. The custom LoRA weights are not included; train them in Notebook 02.

**Medical caution:** the optional texture LoRA is not clinically validated. Its published trigger mentions acne; do not assume this adapter improves normal skin or preserves absence of lesions. Generated images are research-only until reviewed.

## 1. Install dependencies — fresh Kaggle session
Select a T4 GPU, enable Internet and run this before importing Diffusers.

In [ ]:
import subprocess, sys
# Keep Kaggle's GPU-enabled PyTorch installation. Restart the kernel after installing if necessary.
PINNED = ['diffusers==0.35.1', 'transformers==4.56.2', 'accelerate==1.10.1',
          'peft==0.17.1', 'huggingface_hub==0.34.4', 'safetensors==0.6.2',
          'ftfy==6.3.1', 'compel==2.3.1']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PINNED])
print('Installed pinned dependencies. Restart the kernel if packages were imported earlier.')

In [ ]:
from pathlib import Path
import gc, hashlib, json, re, time, zipfile, shutil
import torch, diffusers, transformers, compel
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler, AutoencoderKL
from compel import CompelForSDXL
from huggingface_hub import model_info
from PIL import Image, ImageDraw
from IPython.display import display
assert torch.cuda.is_available(), 'Enable Kaggle T4 GPU under Notebook Settings → Accelerator.'
print('Torch:', torch.__version__, 'Diffusers:', diffusers.__version__, 'Transformers:', transformers.__version__)
print('Compel:', getattr(compel, '__version__', '2.3.1 (version attribute unavailable)'))
print('GPU:', torch.cuda.get_device_name(0))

## 2. Edit generation settings

`MODES=['base']` is the safe initial SDXL baseline. To compare the *published optional LoRA* before training, use `['base','texture']`. After training Notebook 02, set `CUSTOM_LORA_SOURCE` to your `.safetensors` file or LoRA directory, then use `['base','custom']` or all four modes. The same five seeds are used in each mode. **The `texture` and `combined` modes are exploratory only:** the public texture adapter is acne-oriented, not a medically validated normal-skin model.

`OUT_DIR` must be new for each run so images from different configurations are never mixed.

In [ ]:
BASE_MODEL = 'stabilityai/stable-diffusion-xl-base-1.0'
VAE_MODEL = 'madebyollin/sdxl-vae-fp16-fix'  # more stable fp16 SDXL decoding
TEXTURE_LORA = 'profpeng/skinenhancer'       # optional aesthetics adapter; includes an acne-oriented trigger
CUSTOM_LORA_SOURCE = ''                     # train with Notebook 02; path/file/Hub ID of SDXL-compatible LoRA
MODES = ['base']                            # allowed: 'base', 'texture', 'custom', 'combined'
TEXTURE_WEIGHT = 0.35                       # experimental; not clinically validated
CUSTOM_WEIGHT = 0.8                         # experimental
PROMPT_MAX_CONTENT_TOKENS = 150
IMAGES_PER_PROMPT = 5
WIDTH = HEIGHT = 768                       # 768 is a T4-friendly pilot; 1024 may need more VRAM
STEPS = 25
GUIDANCE_SCALE = 6.0
START_SEED = 561231
NEGATIVE_PROMPT = ('cartoon, drawing, CGI, illustration, overprocessed skin, airbrushing, '
                   'beauty filter, text, watermark, plastic skin, deformed face, duplicate face, '
                   'unnatural features, severe blur')
OUT_DIR = Path('/kaggle/working/normal_skin_generated')
ALLOWED_MODES = {'base', 'texture', 'custom', 'combined'}
assert MODES and set(MODES) <= ALLOWED_MODES and len(MODES)==len(set(MODES))
assert WIDTH % 8 == HEIGHT % 8 == 0 and IMAGES_PER_PROMPT == 5
if any(mode in MODES for mode in ['custom','combined']) and not CUSTOM_LORA_SOURCE.strip():
    raise ValueError('Set CUSTOM_LORA_SOURCE to the SDXL normal-face LoRA from Notebook 02.')
print('Modes:', MODES, '| images per prompt:', 5*len(MODES))

## 3. Authored morphology prompts
Uses the project `prompts/` JSON files, 300 per class with protected group-disjoint train/validation/test prompt families. Defaults to a five-prompt **train-only** pilot and uses each prompt's original negative text. Change `PROMPT_LIMIT=None` to run all 210 train prompts. Outputs include a review-draft CSV; all generated images begin **unapproved**.


In [ ]:
# Split-safe authored prompts (210 train, 45 val, 45 test; 60 disjoint families).
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import prompt_collection,generation_manifest,append_run_log
PROMPT_PARTITION='train'  # DO NOT augment with val/test prompt families
PROMPT_OFFSET=0
PROMPT_LIMIT=5  # Kaggle pilot; None for all 210 training prompts
PROMPTS=prompt_collection(DERMSYNREAL_REPO,'Normal',partition=PROMPT_PARTITION,limit=PROMPT_LIMIT,offset=PROMPT_OFFSET)
print('Authored prompts:',len(PROMPTS),'families:',len({p['prompt_family_id'] for p in PROMPTS}))

## 4. Download SDXL and activate any requested LoRA adapters
Model revisions are recorded. **Only SDXL LoRAs may be loaded**. No SD2.1 disease textual-inversion model belongs here.

In [ ]:
base_sha = model_info(BASE_MODEL).sha
vae_sha = model_info(VAE_MODEL).sha
texture_sha = model_info(TEXTURE_LORA).sha if any(m in MODES for m in ('texture','combined')) else None
print('Base revision:', base_sha, 'VAE revision:', vae_sha, 'Texture revision:', texture_sha)

vae = AutoencoderKL.from_pretrained(VAE_MODEL, revision=vae_sha, torch_dtype=torch.float16, use_safetensors=True)
pipe = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL, revision=base_sha, variant='fp16', use_safetensors=True,
    torch_dtype=torch.float16, vae=vae,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

if any(m in MODES for m in ('texture','combined')):
    pipe.load_lora_weights(TEXTURE_LORA, adapter_name='texture')
    print('Optional texture LoRA loaded; research control ONLY (not a medical normal-skin guarantee).')
if any(m in MODES for m in ('custom','combined')):
    source = CUSTOM_LORA_SOURCE.strip()
    local = Path(source)
    if local.is_file():
        if local.suffix != '.safetensors':
            raise ValueError('CUSTOM_LORA_SOURCE file must be .safetensors')
        pipe.load_lora_weights(str(local.parent), weight_name=local.name, adapter_name='normal_face')
    else:
        pipe.load_lora_weights(source, adapter_name='normal_face')
    print('Loaded custom SDXL LoRA:', source)
    # Verify that notebook 02 used the same SDXL base. Missing manifests are allowed
    # for externally trained LoRAs, but are explicitly disclosed.
    provenance = local.parent / 'training_provenance.json' if local.is_file() else local / 'training_provenance.json'
    if provenance.is_file():
        info = json.loads(provenance.read_text(encoding='utf-8'))
        if info.get('base_model') != BASE_MODEL:
            raise ValueError(f"Custom LoRA base mismatch: {info.get('base_model')} != {BASE_MODEL}")
        if info.get('base_commit') and info['base_commit'] != base_sha:
            raise ValueError('LoRA base revision differs from the pinned inference base revision')
        print('Custom LoRA provenance matches SDXL base/revision')
    else:
        print('WARNING: Custom LoRA has no training_provenance.json; architecture/revision cannot be verified automatically')

pipe.to('cuda')
pipe.set_progress_bar_config(disable=False)
print('SDXL pipeline initialized')

## 5. Validate 150 content tokens using both actual SDXL tokenizers

Each CLIP tokenizer has 77 positions per chunk (75 content positions plus special tokens). A target of at most **150 content tokens in each tokenizer** means at most two chunks. Both positive and negative prompts are validated before generation; 151 content tokens causes an explicit error. Compel 2.3.1's `CompelForSDXL` wrapper uses **non-truncating long-prompt encoding by default** (as documented by its authors), with padded positive/negative conditioning and SDXL-specific pooled embeddings. Their shapes are checked before inference. This does not guarantee every medical morphology instruction is followed.

In [ ]:
def content_count(tokenizer, text):
    ids = tokenizer(text, truncation=False, add_special_tokens=True)['input_ids']
    return max(0, len(ids)-2)

def validate_prompt(text, prompt_name='prompt'):
    for name, tokenizer in [('tokenizer', pipe.tokenizer), ('tokenizer_2',pipe.tokenizer_2)]:
        if int(tokenizer.model_max_length) != 77:
            raise ValueError(f'Expected 77-position CLIP context for {name}; got {tokenizer.model_max_length}')
        n = content_count(tokenizer, text)
        if n > PROMPT_MAX_CONTENT_TOKENS:
            raise ValueError(f'{prompt_name}: {n} content tokens in {name} exceeds {PROMPT_MAX_CONTENT_TOKENS}; please shorten.')
        print(f'{prompt_name} | {name}: {n}/{PROMPT_MAX_CONTENT_TOKENS} content tokens')

for entry in PROMPTS:
    validate_prompt(entry['text'], entry['id'])
for entry in PROMPTS:
    validate_prompt(entry.get('negative_prompt',NEGATIVE_PROMPT),entry['id']+'/negative')

encoder = CompelForSDXL(pipe)   # Compel 2.3.1: truncation OFF by default for this SDXL wrapper

def encode_pair(positive, negative):
    with torch.inference_mode():
        result = encoder(prompt=positive, negative_prompt=negative)
    p = result.embeds
    n = result.negative_embeds
    pp = result.pooled_embeds
    npool = result.negative_pooled_embeds
    if p.shape != n.shape or p.shape[1] > 154 or p.shape[1] % 77 != 0:
        raise ValueError(f'Invalid Compel conditioning lengths: positive={tuple(p.shape)}, negative={tuple(n.shape)}; 2 chunks maximum.')
    if pp.shape != npool.shape or pp.shape[0] != p.shape[0]:
        raise ValueError('Positive and negative pooled conditioning shapes do not match.')
    return p, n, pp, npool

## 6. Five images per prompt, matching seeds across modes, metadata, contact-sheet previews
`mode` controls LoRA weights. Base mode disables LoRA explicitly rather than silently using an active adapter. Experimental texture mode may change normal morphology; reject any abnormal output after review.

In [ ]:
def select_mode(mode):
    if mode == 'base':
        if any(m in MODES for m in ('texture','custom','combined')):
            pipe.disable_lora()
        return
    pipe.enable_lora()
    if mode == 'texture':
        pipe.set_adapters(['texture'], adapter_weights=[TEXTURE_WEIGHT])
    elif mode == 'custom':
        pipe.set_adapters(['normal_face'], adapter_weights=[CUSTOM_WEIGHT])
    elif mode == 'combined':
        pipe.set_adapters(['texture','normal_face'], adapter_weights=[TEXTURE_WEIGHT,CUSTOM_WEIGHT])
    else:
        raise ValueError(f'Unsupported mode: {mode}')

def safe(s):
    return re.sub(r'[^a-zA-Z0-9_-]+','_',s).strip('_')[:60] or 'prompt'

if OUT_DIR.exists():
    raise FileExistsError(f'{OUT_DIR} already exists. Choose a new OUT_DIR to avoid mixing two generations.')
OUT_DIR.mkdir(parents=True)
rows=[]; t0=time.time()
for i, entry in enumerate(PROMPTS):
    for mode in MODES:
        select_mode(mode)
        # Encode AFTER choosing LoRA mode so text-encoder LoRAs, if present,
        # cannot silently reuse embeddings from a different mode.
        p,n,pp,npool = encode_pair(entry['text'], entry.get('negative_prompt',NEGATIVE_PROMPT))
        folder = OUT_DIR / safe(entry['id']) / mode
        folder.mkdir(parents=True, exist_ok=False)
        images=[]
        for j in range(IMAGES_PER_PROMPT):
            seed=START_SEED+1000*i+j    # matched seeds in each mode
            generator=torch.Generator(device='cuda').manual_seed(seed)
            with torch.inference_mode():
                img=pipe(prompt_embeds=p, negative_prompt_embeds=n,
                         pooled_prompt_embeds=pp, negative_pooled_prompt_embeds=npool,
                         height=HEIGHT, width=WIDTH, num_inference_steps=STEPS,
                         guidance_scale=GUIDANCE_SCALE, generator=generator).images[0]
            fp=folder/f'{safe(entry["id"])}_{mode}_seed{seed}.png'
            img.save(fp)
            images.append(img)
            rows.append({'file':str(fp.relative_to(OUT_DIR)), 'prompt_id':entry['id'],
                         'class':'normal', 'mode':mode,'prompt':entry['text'],
                         'negative_prompt':entry.get('negative_prompt',NEGATIVE_PROMPT),'seed':seed,'steps':STEPS,
                         'guidance_scale':GUIDANCE_SCALE,'width':WIDTH,'height':HEIGHT,
                         'base_model':BASE_MODEL,'base_commit':base_sha,'vae_model':VAE_MODEL,
                         'vae_commit':vae_sha,'texture_lora':TEXTURE_LORA if mode in ('texture','combined') else None,
                         'texture_commit':texture_sha if mode in ('texture','combined') else None,
                         'custom_lora':CUSTOM_LORA_SOURCE if mode in ('custom','combined') else None,
                         'custom_lora_weight':CUSTOM_WEIGHT if mode in ('custom','combined') else None,
                         'texture_lora_weight':TEXTURE_WEIGHT if mode in ('texture','combined') else None,
                         'prompt_max_content_tokens':PROMPT_MAX_CONTENT_TOKENS,
                         'clinical_review':'pending', 'duplicate_check':'pending'})
        montage=Image.new('RGB',(WIDTH*IMAGES_PER_PROMPT,HEIGHT))
        for j,img in enumerate(images): montage.paste(img, (j*WIDTH,0))
        display(montage.resize((min(1200,montage.width), max(1,round(montage.height*min(1200,montage.width)/montage.width)))))
        del p,n,pp,npool
        gc.collect()

with (OUT_DIR/'metadata.jsonl').open('w', encoding='utf-8') as handle:
    for row in rows: handle.write(json.dumps(row,ensure_ascii=False)+'\n')
assert len(rows)==len(PROMPTS)*IMAGES_PER_PROMPT*len(MODES)
print('Generated',len(rows),'images in',round((time.time()-t0)/60,1),'min')

In [ ]:
# Write Kaggle CSV/JSON audit outputs BEFORE packaging the generated images.
summary=generation_manifest(rows,OUT_DIR/'reports',root=OUT_DIR,class_name='Normal',repo_root=DERMSYNREAL_REPO,prompt_partition=PROMPT_PARTITION,model_id=BASE_MODEL)
append_run_log(OUT_DIR,kind='generation',config={'class':'Normal','prompt_partition':PROMPT_PARTITION,'prompt_count':len(PROMPTS),'images_per_prompt':IMAGES_PER_PROMPT,'generated':summary['generated_images']},artifacts=[OUT_DIR/'reports/generation_manifest.csv',OUT_DIR/'metadata.jsonl'],repo_root=DERMSYNREAL_REPO)
print('Review draft:',OUT_DIR/'reports/synthetic_pool_review_draft.csv','| Run log:',OUT_DIR/'run_log.jsonl')

## 7. Export outputs to ZIP — Kaggle Output panel

In [ ]:
ZIP_PATH=Path('/kaggle/working/normal_skin_images.zip')
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for path in sorted(OUT_DIR.rglob('*')):
        if path.is_file(): z.write(path, arcname=str(path.relative_to(OUT_DIR)))
print('ZIP:',ZIP_PATH, '| files:', len(rows))

## Research caution
Generated normal faces can exhibit subtle signs of acne, pigmentation, or retouching artifacts. Do not accept every image as a verified healthy normal skin sample. Review morphology and perform duplicate/near-copy and identity-protection checks against real training data. The optional public skin texture LoRA was not validated for clinical normal skin.